# Example: Static forward free-boundary equilibrium calculations (in OpenSTEP)

---

Here we set up the [OpenSTEP](https://github.com/ukaea/OpenSTEP) tokamak, plot its coil and limiter geometry, and solve a static forward free-boundary Grad-Shafranov equilibrium using the packaged coil currents and profile data.

Refer back to **Example02c** where we analyse convergence of this case using higher resolution spatial meshes - this is required for STEP-like plasmas that have quite complex internal current density profiles!

### Import packages

In [ ]:
import pickle
import matplotlib.pyplot as plt
import numpy as np

### Create the machine object

In [ ]:
from freegsnke import build_machine

tokamak = build_machine.tokamak(
    active_coils_path="../machine_configs/OpenSTEP/OpenSTEP_simple_active_coils.pickle",
    limiter_path="../machine_configs/OpenSTEP/OpenSTEP_simple_limiter.pickle",
    wall_path="../machine_configs/OpenSTEP/OpenSTEP_simple_limiter.pickle",
)

In [ ]:
# plot the machine
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
plt.tight_layout()

tokamak.plot(axis=ax1, show=False)
ax1.fill(tokamak.wall.R, tokamak.wall.Z, color='k', linewidth=1.2, facecolor='w', zorder=0)

ax1.grid(zorder=0, alpha=0.5)
ax1.set_aspect('equal')
ax1.set_xlabel(r'Major radius, $R$ [m]')
ax1.set_ylabel(r'Height, $Z$ [m]')

### Instantiate an equilibrium

Let's set up the equilibrium and profiles objects using the packaged data. We'll then design an initial plasma flux guess using some approxiamte information about the plasma shape. 

In [ ]:
# load the p' and FF' profile data from file
with open('../machine_configs/OpenSTEP/OpenSTEP_profiles.pickle', 'rb') as f:
    profiles_data = pickle.load(f)

In [ ]:
from freegsnke import equilibrium_update
from freegsnke.jtor_update import GeneralPprimeFFprime
from freegsnke.equilibrium import PsiGuessParabolicJtor

# equilibrium
eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,        # provide tokamak object
    Rmin=0.5, Rmax=9.0,     # radial range
    Zmin=-10.0, Zmax=10.0,  # vertical range
    nx=65,                  # number of grid points in the radial direction
    ny=129,                 # number of grid points in the vertical direction
)

# set active coil currents
currentvec = np.array([
    -4790870.39240653,
     5155466.64491291,
     2254704.04332795,
     5458947.31403196,
    -7854367.88994826,
     5100000.0,
    -2274646.53005716,
])
eq.tokamak.set_all_coil_currents(currentvec)


# profiles
profiles = GeneralPprimeFFprime(
    eq=eq,
    Ip=2.2767e7,
    fvac=0.5,
    psi_n=np.linspace(0,1,256),
    pprime_data=profiles_data['pprime'],
    ffprime_data=profiles_data['ffprime']
)

# create the initial guess using alternate method
parabolic_plasma_psi = PsiGuessParabolicJtor(
    Ip = 2.2767e7,    # plasma current (to scale the guess)
    h = (max(tokamak.wall.R) + min(tokamak.wall.R))/2,   # centre of ellipse (R)
    k = 0.0,                                             # centre of ellipse (Z)
    a = 1.0,          # roughly the minor radius of the plasma
    b = 1.0*5,        # roughly the minor radius times the elongation of expected plasma
    psi_scale = 1.0   # might need to vary this default
)

# set plasma psi guess
_plasma_psi = parabolic_plasma_psi(eq=eq) 
eq.plasma_psi = _plasma_psi

### Load the static nonlinear solver

In [ ]:
from freegsnke import GSstaticsolver

GSStaticSolver = GSstaticsolver.NKGSsolver(
    eq=eq,
    solver_type='LUsparse',
    gs_operator_order=4,
)

### The forward solve

Note that we solve to a high tolerance and the `force_up_down_symmetric` flag has been enabled. This is because of the complexity of the internal plasma current density profile of this equilibrium.

In [ ]:
# solver
GSStaticSolver.solve(
    eq=eq,
    profiles=profiles,
    constrain=None,
    target_relative_tolerance=1e-10,
    force_up_down_symmetric=True,
    verbose=True,
)

In [ ]:
# plot the converged equilibrium
fig2, ax2 = plt.subplots(1, 1, figsize=(5, 10), dpi=80)
plt.tight_layout()

ax2.grid(zorder=0, alpha=0.5)
ax2.set_aspect('equal')
eq.tokamak.plot(axis=ax2, show=False)
ax2.fill(tokamak.wall.R, tokamak.wall.Z, color='k', linewidth=1.2, facecolor='w', zorder=0)
eq.plot(axis=ax2, show=False)
ax2.set_xlabel(r'Major radius, $R$ [m]')
ax2.set_ylabel(r'Height, $Z$ [m]')

In [ ]:
eq.tokamak.getCurrents()